# CASMI 2026 - Tier-0 Diagnostics

Six measurements that decide where the score actually is. **Run this before changing the
baseline notebook.** Each section prints a `VERDICT:` line you can act on directly.

| # | Question | Decides |
|---|---|---|
| D4 | How far is each library's `precursor_mz`-derived neutral mass from its *label's* exact mass? | Whether `riken` (347k spectra) is invisible to Channel 1 |
| D1 | What fraction of true structures are even inside the candidate mass window? | The hard MRR ceiling; database expansion vs. ranking |
| D1b | Is the mass window mis-centred by a systematic instrument offset? | Whether free self-calibration is worth it |
| D6 | Does raising `INT_FLOOR` / cutting `MAX_PEAKS` / structure-indexing help? | Three config changes, measured against a published benchmark |
| D3 | Is MetFrag-lite's explain score saturated, or disabled by `max_bonds=34`? | Whether Channel 3 is noise |
| D5 | How many of the 25 slots collapse to duplicate tautomer-canonical InChIKey14? | Free score from deduplication |

**Inputs required** - attach exactly what the baseline notebook attaches: the competition data,
the RDKit wheel, and the COCONUT pack (`coco_fp.npy`, `coco_mass.npy`, `coco_meta.pkl`).

Query set throughout is `enveda-np-examples`: 250 natural products on the **test instrument**
through the **test pipeline**. It is the organisers' own bridge between train and test chemistry,
and the only honest cheap proxy available. The published reference point for a plain
modified-cosine library search on this exact set is **MRR@25 = 0.9090**.


In [1]:
import os, sys, re, glob, time, math, pickle, collections, subprocess, warnings
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
T0 = time.time()

def find_file(name):
    hits = glob.glob("/kaggle/input/**/" + name, recursive=True)
    if not hits: raise FileNotFoundError(name)
    return sorted(hits, key=len)[0]

COMP  = os.path.dirname(find_file("test.parquet"))
TRAIN = os.path.join(COMP, "train.parquet")
TEST  = os.path.join(COMP, "test.parquet")
print("competition dir:", COMP)

whl = glob.glob("/kaggle/input/**/rdkit-*.whl", recursive=True)
if whl:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", whl[0]], check=False)
try:
    from rdkit import Chem, RDLogger
    from rdkit.Chem.MolStandardize import rdMolStandardize
    RDLogger.DisableLog("rdApp.*")
    HAVE_RDKIT = True
except Exception as e:
    HAVE_RDKIT = False
    print("[WARN] RDKit unavailable:", e)
print("rdkit:", HAVE_RDKIT)

# ---- adduct physics, identical to the baseline notebook ----
MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,
            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,
            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)
E = 0.00054857990
PROTON = MASS["H"] - E
H2O = 2 * MASS["H"] + MASS["O"]; NH4 = MASS["N"] + 4 * MASS["H"]
FORMATE = MASS["C"] + 2 * MASS["H"] + 2 * MASS["O"]
ACETATE = 2 * MASS["C"] + 4 * MASS["H"] + 2 * MASS["O"]
ADDUCTS = {
 "[M+H]+": (1,1,PROTON), "[M+NH4]+": (1,1,NH4-E), "[M+Na]+": (1,1,MASS["Na"]-E),
 "[M+K]+": (1,1,MASS["K"]-E), "[M-H2O+H]+": (1,1,PROTON-H2O), "[M-2H2O+H]+": (1,1,PROTON-2*H2O),
 "[M+2H]2+": (1,2,2*PROTON), "[M]+": (1,1,-E), "[M-H2O]+": (1,1,-E-H2O),
 "[M+CH3OH+H]+": (1,1,PROTON+MASS["C"]+4*MASS["H"]+MASS["O"]),
 "[M+CH3CN+H]+": (1,1,PROTON+2*MASS["C"]+3*MASS["H"]+MASS["N"]),
 "[M-H]-": (1,1,-PROTON), "[M-H2O-H]-": (1,1,-PROTON-H2O), "[M+CH2O2-H]-": (1,1,FORMATE-PROTON),
 "[M+C2H4O2-H]-": (1,1,ACETATE-PROTON), "[M+Cl]-": (1,1,MASS["Cl"]+E), "[M]-": (1,1,E),
 "[M-2H]-": (1,2,-2*PROTON), "[M+Na-2H]-": (1,1,MASS["Na"]-2*PROTON),
 "[2M+H]+": (2,1,PROTON), "[2M+Na]+": (2,1,MASS["Na"]-E), "[2M+NH4]+": (2,1,NH4-E),
 "[2M+K]+": (2,1,MASS["K"]-E), "[2M-H]-": (2,1,-PROTON), "[2M+CH2O2-H]-": (2,1,FORMATE-PROTON),
 "[2M+C2H4O2-H]-": (2,1,ACETATE-PROTON), "[2M+Na-2H]-": (2,1,MASS["Na"]-2*PROTON),
 "[3M+H]+": (3,1,PROTON), "[3M-H]-": (3,1,-PROTON),
}
def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan)
    ad = np.asarray(adduct, dtype=object)
    for a, (n, z, d) in ADDUCTS.items():
        m = (ad == a)
        if m.any(): out[m] = (mz[m] * z - d) / n
    return out

# ---- exact mass straight from molecular_formula: no RDKit, and far faster than 275k parses ----
MONO = {"C":12.0,"H":1.0078250319,"N":14.0030740052,"O":15.9949146221,"S":31.97207069,
        "P":30.97376151,"F":18.99840320,"Cl":34.96885271,"Br":78.9183376,"I":126.904468,
        "Si":27.9769265327,"Se":79.9165196,"B":11.0093055,"Na":22.98976928,"K":38.96370649,
        "Fe":55.9349375,"D":2.0141017778,"As":74.9215965,"Li":7.0160034,"Sn":119.9021947}
TOK = re.compile(r"([A-Z][a-z]?)(\d*)")
def formula_mass(f):
    if not isinstance(f, str): return np.nan
    s, pos = 0.0, 0
    for m in TOK.finditer(f):
        if m.start() != pos: return np.nan
        pos = m.end()
        el = m.group(1)
        if el not in MONO: return np.nan
        s += MONO[el] * (int(m.group(2)) if m.group(2) else 1)
    return s if pos == len(f) else np.nan

META = ["ingest_lib","inchikey14","normalized_smiles","molecular_formula",
        "ionization_mode","adduct","precursor_mz","num_peaks"]
tr = pq.read_table(TRAIN, columns=META).to_pandas()
tr["gidx"] = np.arange(len(tr), dtype=np.int64)
te = pq.read_table(TEST).to_pandas()
print("train {:,} spectra / {:,} structures | test {:,} spectra"
      .format(len(tr), tr.inchikey14.nunique(), len(te)))

fmass = {f: formula_mass(f) for f in tr.molecular_formula.unique()}
tr["exact_mass"] = tr.molecular_formula.map(fmass)
tr["nm_meas"]    = neutral_mass(tr.precursor_mz.values.astype(np.float64), tr.adduct.values)
print("formula mass resolved for {:.2f}% of rows".format(tr.exact_mass.notna().mean() * 100))
print("adduct recognised for      {:.2f}% of rows".format(tr.nm_meas.notna().mean() * 100))
print("setup done in {:.1f}s".format(time.time() - T0))

competition dir: /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra
rdkit: True
train 2,539,608 spectra / 275,810 structures | test 1,213 spectra
formula mass resolved for 99.54% of rows
adduct recognised for      99.91% of rows
setup done in 11.4s


---
## D4 - How far is the measured neutral mass from the label's exact mass?

The baseline indexes its reference library on `neutral_mass(precursor_mz, adduct)`. If a library
reports `precursor_mz` coarsely, its spectra land outside the `+/-8.5 ppm` Channel-1 window and
become **unreachable**, however good the spectral match would have been.

The column that matters is the last one: *the share of each library Channel 1 can actually see.*


In [2]:
ok = tr.exact_mass.notna() & tr.nm_meas.notna() & (tr.exact_mass > 0)
d  = tr[ok].copy()
d["ppm"] = (d.nm_meas - d.exact_mass) / d.exact_mass * 1e6

def decimals(v):
    s = "{:.10f}".format(v).rstrip("0")
    return len(s.split(".")[1]) if "." in s else 0

rows = []
for lib, g in d.groupby("ingest_lib"):
    a = g.ppm.abs()
    rows.append({
        "ingest_lib": lib,
        "spectra": len(g),
        "median_abs_ppm": round(float(a.median()), 2),
        "p90_abs_ppm":    round(float(a.quantile(.90)), 1),
        "mz_decimals":    int(np.median([decimals(x) for x in g.precursor_mz.head(3000)])),
        "within_3ppm_pct": round(float((a <= 3).mean() * 100), 1),
        "VISIBLE_at_8p5ppm_pct": round(float((a <= 8.5).mean() * 100), 1),
        "within_30ppm_pct": round(float((a <= 30).mean() * 100), 1),
    })
D4 = pd.DataFrame(rows).sort_values("VISIBLE_at_8p5ppm_pct")
print(D4.to_string(index=False))

lost = d.groupby("ingest_lib").apply(lambda g: int((g.ppm.abs() > 8.5).sum()))
tot_lost = int(lost.sum())
print("\nSpectra outside the +/-8.5 ppm Channel-1 window: {:,} of {:,} ({:.1f}%)"
      .format(tot_lost, len(d), tot_lost / len(d) * 100))
print(lost.sort_values(ascending=False).head(6).to_string())

worst = D4.iloc[0]
print("\nVERDICT: worst library is '{}' at {:.1f}% visible (m/z to {} dp, median |error| {:.1f} ppm)."
      .format(worst.ingest_lib, worst.VISIBLE_at_8p5ppm_pct, worst.mz_decimals, worst.median_abs_ppm))
if worst.VISIBLE_at_8p5ppm_pct < 90:
    print("         -> Idea 1.1 is live. Index the library on the LABEL's exact mass, not on")
    print("            precursor_mz. Cost ~1 hour; recovers the lost spectra outright.")
else:
    print("         -> Idea 1.1 is NOT the problem here. Skip it and move to D1.")

        ingest_lib  spectra  median_abs_ppm  p90_abs_ppm  mz_decimals  within_3ppm_pct  VISIBLE_at_8p5ppm_pct  within_30ppm_pct
             riken   341520            3.46         21.8            2             47.3                   70.1              92.4
          massbank    98487            0.11         10.8            4             86.4                   89.7              90.9
              gnps   219285            0.99          4.3            3             84.9                   92.5              94.0
              mona    91750            0.12          3.6            4             88.7                   93.3              94.5
         drug_plus     2514            0.08          0.2            4             96.5                   96.5              96.5
enveda-np-examples     1184            1.68          3.3           10             87.5                   99.5             100.0
            msdial    40354            0.06          0.2            4             99.7                  

---
## D4b - is the loss concentrated in test-relevant chemistry?

A library being invisible only matters in proportion to how much of the *right* chemistry it
holds. `enveda-180` is 45% of the file, but its Jaccard overlap with every natural-product
library is ~0.004, so losing it would cost little. Losing `riken` - plant specialised
metabolites - would cost a great deal.


In [3]:
NP_LIBS = ["riken","mona","msdial","massbank","gnps","enveda-np-examples","masaryk"]
d["is_np"] = d.ingest_lib.isin(NP_LIBS)
for lab, g in [("natural-product libraries", d[d.is_np]), ("enveda-180 + synthetic", d[~d.is_np])]:
    vis = (g.ppm.abs() <= 8.5).mean() * 100
    print("{:28} {:>10,} spectra   {:>8,} structures   {:5.1f}% visible at 8.5 ppm"
          .format(lab, len(g), g.inchikey14.nunique(), vis))

np_all = d[d.is_np].inchikey14.nunique()
np_vis = d[d.is_np & (d.ppm.abs() <= 8.5)].inchikey14.nunique()
print("\nNP structures with at least one visible spectrum: {:,} / {:,} ({:.1f}%)"
      .format(np_vis, np_all, np_vis / np_all * 100))
print("NP structures Channel 1 can NEVER match:          {:,}".format(np_all - np_vis))
print("\nVERDICT: that last number is how many natural products sit in your reference library")
print("         but cannot be retrieved by direct spectral search, for a purely numeric reason.")

natural-product libraries       793,232 spectra     54,650 structures    83.0% visible at 8.5 ppm
enveda-180 + synthetic        1,732,462 spectra    237,796 structures   100.0% visible at 8.5 ppm

NP structures with at least one visible spectrum: 52,249 / 54,650 (95.6%)
NP structures Channel 1 can NEVER match:          2,401

VERDICT: that last number is how many natural products sit in your reference library
         but cannot be retrieved by direct spectral search, for a purely numeric reason.


---
## D1 - Pool recall: the hard MRR ceiling

If the true structure is not inside `pool.window(target, ppm)`, the molecule scores **0** however
good the ranker is. Two separate failure modes needing different fixes:

- **coverage failure** - the structure is not in the database at all -> Idea 2.1 (expand the pool)
- **mass-window failure** - it *is* in the database, but the measured precursor puts the window in
  the wrong place -> Idea 1.5 (self-calibrate the window)

Separating them is the whole point of this cell.


In [4]:
cdir    = os.path.dirname(find_file("coco_fp.npy"))
cmeta   = pickle.load(open(os.path.join(cdir, "coco_meta.pkl"), "rb"))
co_mass = np.load(os.path.join(cdir, "coco_mass.npy"))
co_keys = np.asarray(cmeta["keys"], dtype=object)
co_smi  = dict(zip(co_keys, np.asarray(cmeta["smiles"], dtype=object)))
smi_of  = dict(zip(tr.inchikey14.values, tr.normalized_smiles.values))
print("COCONUT pack: {:,} structures".format(len(co_mass)))

u = tr.dropna(subset=["exact_mass"]).drop_duplicates("inchikey14")
u = u[~u.inchikey14.isin(set(co_keys))]
pool_keys = np.concatenate([co_keys, u.inchikey14.values])
pool_mass = np.concatenate([co_mass, u.exact_mass.values])
fin = np.isfinite(pool_mass.astype(float))
pool_keys, pool_mass = pool_keys[fin], pool_mass[fin].astype(float)
o = np.argsort(pool_mass); pool_keys, pool_mass = pool_keys[o], pool_mass[o]
key_set = set(pool_keys.tolist())
print("unified pool: {:,} structures (COCONUT + train)".format(len(pool_mass)))

# query molecules: enveda-np-examples, aggregated per structure exactly as inference does per molecule_id
q  = tr[tr.ingest_lib == "enveda-np-examples"]
qs = q[q.nm_meas.notna()]
qg = (qs.groupby("inchikey14")
        .agg(target=("nm_meas", "median"), truth=("exact_mass", "first")).reset_index())
qg = qg.dropna(subset=["truth"])
print("query structures: {}".format(len(qg)))

def window_keys(t, ppm):
    a = np.searchsorted(pool_mass, t * (1 - ppm / 1e6), "left")
    b = np.searchsorted(pool_mass, t * (1 + ppm / 1e6), "right")
    return pool_keys[a:b], int(b - a)

in_db = qg.inchikey14.isin(key_set)
print("\nCOVERAGE: true structure present anywhere in the pool: {} / {} ({:.1f}%)"
      .format(int(in_db.sum()), len(qg), in_db.mean() * 100))

rows = []
for ppm in [3, 5, 8.5, 10, 20, 30]:
    hit_meas, hit_true, ncand = 0, 0, []
    for _, r in qg.iterrows():
        k, n = window_keys(r.target, ppm); ncand.append(n)
        if r.inchikey14 in set(k.tolist()): hit_meas += 1
        k2, _ = window_keys(r.truth, ppm)
        if r.inchikey14 in set(k2.tolist()): hit_true += 1
    rows.append({"ppm": ppm,
                 "recall_measured_mass_pct": round(hit_meas / len(qg) * 100, 1),
                 "recall_TRUE_mass_pct":     round(hit_true / len(qg) * 100, 1),
                 "median_candidates": int(np.median(ncand)),
                 "p95_candidates":    int(np.percentile(ncand, 95))})
D1 = pd.DataFrame(rows)
print("\n" + D1.to_string(index=False))

r85 = D1[D1.ppm == 8.5].iloc[0]
gap = r85.recall_TRUE_mass_pct - r85.recall_measured_mass_pct
cov_loss = 100 - in_db.mean() * 100
print("\nVERDICT at the baseline's 8.5 ppm:")
print("  ceiling on MRR from pool recall alone : {:.1f}%".format(r85.recall_measured_mass_pct))
print("  lost to DATABASE COVERAGE             : {:.1f} pts  -> Idea 2.1".format(cov_loss))
print("  lost to MASS-WINDOW placement         : {:.1f} pts  -> Idea 1.5".format(gap))
if gap > 2:
    print("  -> the measured precursor is mis-placing the window. Self-calibration is real score.")
if cov_loss > 10:
    print("  -> database coverage is the binding constraint. Expand the pool BEFORE tuning the ranker.")

COCONUT pack: 436,389 structures
unified pool: 711,710 structures (COCONUT + train)
query structures: 250

COVERAGE: true structure present anywhere in the pool: 250 / 250 (100.0%)

 ppm  recall_measured_mass_pct  recall_TRUE_mass_pct  median_candidates  p95_candidates
 3.0                      87.6                 100.0                 38             217
 5.0                      97.6                 100.0                 48             232
 8.5                      99.2                 100.0                 53             255
10.0                     100.0                 100.0                 58             272
20.0                     100.0                 100.0                 76             320
30.0                     100.0                 100.0                105             429

VERDICT at the baseline's 8.5 ppm:
  ceiling on MRR from pool recall alone : 99.2%
  lost to DATABASE COVERAGE             : 0.0 pts  -> Idea 2.1
  lost to MASS-WINDOW placement         : 0.8 pts  -> I

---
## D1b - is the window mis-centred by a systematic offset?

If the ppm residuals have a non-zero median, the instrument has a calibration offset and every
query is shifted the same way. That is correctable for free at inference time and with no labels,
using only the confident Channel-1 hits (Idea 1.5).

Re-centring beats widening: a wider window adds decoys linearly, a centred one does not.


In [5]:
res = ((qg.target - qg.truth) / qg.truth * 1e6).dropna()
print("ppm residual (measured neutral mass - true exact mass), n={}".format(len(res)))
print("  median {:+.2f} ppm | mean {:+.2f} | IQR [{:+.2f}, {:+.2f}] | p5/p95 [{:+.2f}, {:+.2f}]"
      .format(res.median(), res.mean(), res.quantile(.25), res.quantile(.75),
              res.quantile(.05), res.quantile(.95)))

need_raw = float(res.abs().quantile(0.95))
need_ctr = float((res - res.median()).abs().quantile(0.95))
print("\nhalf-width needed for 95% recall:  as-is {:.1f} ppm   after re-centring {:.1f} ppm"
      .format(need_raw, need_ctr))
print("\nVERDICT: baseline hard-codes PPM_WIN=8.5 with a '+1.4 ppm systematic offset' note.")
print("         Measured offset on this instrument-matched set: {:+.2f} ppm.".format(res.median()))
if abs(res.median()) > 1.0:
    print("         -> non-trivial. Re-centre the window rather than widening it.")
else:
    print("         -> offset is small; the 8.5 ppm setting looks defensible.")

ppm residual (measured neutral mass - true exact mass), n=250
  median +1.45 ppm | mean +1.45 | IQR [+0.56, +2.20] | p5/p95 [-1.05, +3.61]

half-width needed for 95% recall:  as-is 3.6 ppm   after re-centring 3.0 ppm

VERDICT: baseline hard-codes PPM_WIN=8.5 with a '+1.4 ppm systematic offset' note.
         Measured offset on this instrument-matched set: +1.45 ppm.
         -> non-trivial. Re-centre the window rather than widening it.


---
## D6 - Intensity floor, peak cap, and structure-derived indexing

Three changes measured at once on the Class-1 proxy, against the published **MRR@25 = 0.9090**:

1. `INT_FLOOR` - baseline uses 0.002 (0.2%); the test set is ~82% noise below 1%
2. `MAX_PEAKS` - baseline uses 256; the 0.9090 reference used 64
3. **indexing** - `precursor_mz`-derived mass (what the baseline does) vs. the label's exact mass
   (Idea 1.1)

Peak extraction is one streaming pass and is the slow part; the eight scoring runs after it are
cheap. Expect roughly 5-12 minutes total.


In [6]:
GATHER_PPM = 20.0
ref = tr[(tr.ingest_lib != "enveda-np-examples") & tr.nm_meas.notna() & tr.exact_mass.notna()]

idx = {}
for variant, masscol in [("precursor", "nm_meas"), ("structure", "exact_mass")]:
    byad = {}
    for ad, g in ref.groupby("adduct"):
        m = g[masscol].values.astype(np.float64)
        o = np.argsort(m)
        byad[ad] = (m[o], g.gidx.values[o], g.inchikey14.values[o])
    idx[variant] = byad

cand_of = {"precursor": {}, "structure": {}}
need = set(qs.gidx.tolist())
for variant in cand_of:
    byad = idx[variant]
    for gq, ad, t in zip(qs.gidx.values, qs.adduct.values, qs.nm_meas.values):
        if ad not in byad:
            cand_of[variant][gq] = (np.empty(0, np.int64), np.empty(0, object)); continue
        m, gi, ik = byad[ad]
        dlt = t * GATHER_PPM / 1e6
        a, b = np.searchsorted(m, t - dlt), np.searchsorted(m, t + dlt)
        cand_of[variant][gq] = (gi[a:b], ik[a:b])
        need.update(gi[a:b].tolist())
print("reference rows to extract: {:,} ({:.1f}% of train)".format(len(need), len(need) / len(tr) * 100))

t0 = time.time()
pf = pq.ParquetFile(TRAIN)
offs = np.cumsum([0] + [pf.metadata.row_group(i).num_rows for i in range(pf.metadata.num_row_groups)])
PEAKS = {}
for rg in range(pf.metadata.num_row_groups):
    lo, hi = offs[rg], offs[rg + 1]
    loc = np.array(sorted(i - lo for i in need if lo <= i < hi), dtype=np.int64)
    if len(loc) == 0: continue
    tbl = pf.read_row_group(rg, columns=["ms2_mzs", "ms2_normalized_intensities"])
    mzc = tbl.column("ms2_mzs").combine_chunks()
    inc = tbl.column("ms2_normalized_intensities").combine_chunks()
    mo, mv = mzc.offsets.to_numpy(), mzc.values.to_numpy()
    io_, iv = inc.offsets.to_numpy(), inc.values.to_numpy()
    for k in loc:
        # .copy() matters: a bare slice pins the whole row-group buffer in memory
        PEAKS[int(k + lo)] = (mv[mo[k]:mo[k+1]].copy(), iv[io_[k]:io_[k+1]].copy())
    del tbl, mzc, inc, mv, iv
print("extracted {:,} peak lists in {:.1f}s".format(len(PEAKS), time.time() - t0))

reference rows to extract: 116,067 (4.6% of train)
extracted 116,067 peak lists in 29.0s


In [7]:
def prep(mz, it, pmz, floor, topn):
    k = (it >= floor) & (mz <= pmz + 2)
    mz, it = mz[k], it[k]
    if len(mz) > topn:
        o = np.argsort(it)[-topn:]; mz, it = mz[o], it[o]
    o = np.argsort(mz)
    v = np.sqrt(it[o]); n = np.linalg.norm(v)
    return mz[o], (v / n if n > 0 else v)

def modified_cosine(a, b, pa, pb, tol=0.02):
    (ma, ia), (mb, ib) = a, b
    if len(ma) == 0 or len(mb) == 0: return 0.0
    dm = mb[None, :] - ma[:, None]
    cand = []
    for M in (np.abs(dm) <= tol, np.abs(dm - (pb - pa)) <= tol):
        ii, jj = np.nonzero(M)
        if len(ii): cand.append(np.stack([ii, jj, ia[ii] * ib[jj]], 1))
    if not cand: return 0.0
    c = np.concatenate(cand); c = c[np.argsort(-c[:, 2])]
    ua, ub, s = set(), set(), 0.0
    for i, j, v in c:
        i, j = int(i), int(j)
        if i in ua or j in ub: continue
        ua.add(i); ub.add(j); s += v
    return float(s)

prec_of = dict(zip(tr.gidx.values, tr.precursor_mz.values))
gmax = int(tr.gidx.max()) + 1
refmass = {}
for variant, col in [("precursor", "nm_meas"), ("structure", "exact_mass")]:
    arr = np.full(gmax, np.nan)
    arr[tr.gidx.values] = tr[col].values.astype(float)
    refmass[variant] = arr

def run(variant, floor, topn, ppm):
    Rc, per_mol = {}, {}
    for gq, ikq, pmz, tgt in zip(qs.gidx.values, qs.inchikey14.values,
                                 qs.precursor_mz.values, qs.nm_meas.values):
        dd = per_mol.setdefault(ikq, {})
        gi, ik = cand_of[variant][gq]
        if len(gi) == 0: continue
        keep = np.abs(refmass[variant][gi] - tgt) <= tgt * ppm / 1e6
        gi, ik = gi[keep], ik[keep]
        if len(gi) == 0: continue
        qa = prep(PEAKS[gq][0], PEAKS[gq][1], pmz, floor, topn)
        for g, k in zip(gi, ik):
            if g not in Rc:
                Rc[g] = prep(PEAKS[g][0], PEAKS[g][1], prec_of[g], floor, topn)
            s = modified_cosine(qa, Rc[g], pmz, prec_of[g])
            if s > dd.get(k, 0.0): dd[k] = s
    rr, t1, t25, nc = [], 0, 0, []
    for ik, dd in per_mol.items():
        nc.append(len(dd))
        ranked = sorted(dd.items(), key=lambda kv: -kv[1])[:25]
        pos = next((i + 1 for i, (k, _) in enumerate(ranked) if k == ik), None)
        rr.append(1.0 / pos if pos else 0.0); t1 += (pos == 1); t25 += (pos is not None)
    n = max(len(per_mol), 1)
    return dict(MRR25=round(float(np.mean(rr)), 4), top1_pct=round(t1 / n * 100, 1),
                top25_pct=round(t25 / n * 100, 1), med_cand=int(np.median(nc)) if nc else 0)

out = []
for variant in ["precursor", "structure"]:
    for floor in [0.002, 0.01]:
        for topn in [64, 256]:
            t0 = time.time()
            r = run(variant, floor, topn, 8.5)
            r.update(indexing=variant, INT_FLOOR=floor, MAX_PEAKS=topn, sec=round(time.time() - t0, 1))
            out.append(r); print(r, flush=True)

D6 = pd.DataFrame(out)[["indexing","INT_FLOOR","MAX_PEAKS","MRR25","top1_pct","top25_pct","med_cand","sec"]]
print("\n" + D6.sort_values("MRR25", ascending=False).to_string(index=False))

base = D6[(D6.indexing == "precursor") & (D6.INT_FLOOR == 0.002) & (D6.MAX_PEAKS == 256)].MRR25.iloc[0]
best = D6.sort_values("MRR25", ascending=False).iloc[0]
print("\nVERDICT: baseline-equivalent config scores MRR@25 = {:.4f}".format(base))
print("         best is {} indexing / floor {} / top {} = {:.4f}  ({:+.4f})"
      .format(best.indexing, best.INT_FLOOR, best.MAX_PEAKS, best.MRR25, best.MRR25 - base))
print("         published reference point for this query set: 0.9090")

{'MRR25': 0.9154, 'top1_pct': 86.8, 'top25_pct': 98.8, 'med_cand': 10, 'indexing': 'precursor', 'INT_FLOOR': 0.002, 'MAX_PEAKS': 64, 'sec': 26.1}
{'MRR25': 0.9122, 'top1_pct': 86.4, 'top25_pct': 98.8, 'med_cand': 10, 'indexing': 'precursor', 'INT_FLOOR': 0.002, 'MAX_PEAKS': 256, 'sec': 44.4}
{'MRR25': 0.909, 'top1_pct': 85.6, 'top25_pct': 98.8, 'med_cand': 10, 'indexing': 'precursor', 'INT_FLOOR': 0.01, 'MAX_PEAKS': 64, 'sec': 20.5}
{'MRR25': 0.908, 'top1_pct': 85.6, 'top25_pct': 98.8, 'med_cand': 10, 'indexing': 'precursor', 'INT_FLOOR': 0.01, 'MAX_PEAKS': 256, 'sec': 26.2}
{'MRR25': 0.9216, 'top1_pct': 88.4, 'top25_pct': 98.4, 'med_cand': 10, 'indexing': 'structure', 'INT_FLOOR': 0.002, 'MAX_PEAKS': 64, 'sec': 28.9}
{'MRR25': 0.9143, 'top1_pct': 87.2, 'top25_pct': 98.4, 'med_cand': 10, 'indexing': 'structure', 'INT_FLOOR': 0.002, 'MAX_PEAKS': 256, 'sec': 48.8}
{'MRR25': 0.9112, 'top1_pct': 86.4, 'top25_pct': 98.4, 'med_cand': 9, 'indexing': 'structure', 'INT_FLOOR': 0.01, 'MAX_PEAKS'

---
## D3 - Is MetFrag-lite (Channel 3) actually discriminating?

Two suspected defects:

- `max_bonds=34` makes `fragment_masses` return **only the intact parent mass** for larger
  molecules. Test masses run to 1,159 Da, so the channel may be switched off exactly where the
  problem is hardest.
- `h_shifts=(-2,-1,0,1,2)` with an absolute `tol=0.01 Da` (50 ppm at 200 Da) may let almost any
  candidate explain almost any peak.

The test: does the score separate **true** candidates from **decoys** in the same mass window?


In [8]:
if not HAVE_RDKIT:
    print("RDKit unavailable - skipping D3.")
else:
    AMU = dict(MASS)
    H_ATOM = AMU["H"]; PROTON_MASS = H_ATOM - 0.00054857990

    def mol_graph(smi):
        m = Chem.MolFromSmiles(smi)
        if m is None: return None
        n = m.GetNumAtoms(); w = np.zeros(n)
        for a in m.GetAtoms():
            w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H_ATOM
        if (w == 0).any(): return None
        return w, [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()], n

    def components(n, bonds, drop):
        adj = [[] for _ in range(n)]
        for i, (a, b) in enumerate(bonds):
            if i in drop: continue
            adj[a].append(b); adj[b].append(a)
        seen = np.zeros(n, bool); comps = []
        for s in range(n):
            if seen[s]: continue
            st = [s]; seen[s] = True; cur = [s]
            while st:
                uu = st.pop()
                for v in adj[uu]:
                    if not seen[v]: seen[v] = True; st.append(v); cur.append(v)
            comps.append(cur)
        return comps

    def fragment_masses(smi, max_breaks=2, max_bonds=34):
        g = mol_graph(smi)
        if g is None: return np.zeros(0), "unparsed"
        w, bonds, n = g; nb = len(bonds)
        if nb == 0 or nb > max_bonds: return np.array([w.sum()]), "BAILED"
        out = {w.sum()}
        for i in range(nb):
            for c in components(n, bonds, {i}): out.add(float(w[c].sum()))
        if max_breaks >= 2:
            for i in range(nb):
                for j in range(i + 1, nb):
                    for c in components(n, bonds, {i, j}): out.add(float(w[c].sum()))
        return np.array(sorted(out)), "ok"

    def explain_score(fm, pmz, pint, mode=1.0, tol=0.01, h=(-2, -1, 0, 1, 2)):
        if len(fm) == 0 or len(pmz) == 0: return 0.0
        ion = np.sort(np.concatenate([fm + dh * H_ATOM +
                                      (PROTON_MASS if mode > 0 else -PROTON_MASS) for dh in h]))
        wgt = np.sqrt(np.asarray(pint, float)); tot = wgt.sum()
        if tot <= 0: return 0.0
        i = np.searchsorted(ion, pmz); ok2 = np.zeros(len(pmz), bool)
        for off in (-1, 0):
            k = np.clip(i + off, 0, len(ion) - 1); ok2 |= np.abs(ion[k] - pmz) <= tol
        return float(wgt[ok2].sum() / tot)

    qs_by_ik = {k: g for k, g in qs.groupby("inchikey14")}
    true_s, decoy_s, bailed, ntot = [], [], 0, 0
    for _, r in qg.sample(min(40, len(qg)), random_state=0).iterrows():
        g = qs_by_ik.get(r.inchikey14)
        if g is None or len(g) == 0: continue
        gq = int(g.gidx.values[0])
        mz, it = PEAKS[gq]
        k = it >= 0.002; pmz, pint = mz[k], it[k]
        mode = 1.0 if g.ionization_mode.iloc[0] == "positive" else -1.0
        kw, _ = window_keys(r.target, 8.5)
        kw = list(kw)[:60]
        if r.inchikey14 not in kw: kw.append(r.inchikey14)
        for kk in kw:
            s = co_smi.get(kk) or smi_of.get(kk)
            if not s: continue
            fm, st = fragment_masses(s); ntot += 1; bailed += (st == "BAILED")
            sc = explain_score(fm, pmz, pint, mode)
            (true_s if kk == r.inchikey14 else decoy_s).append(sc)

    true_s, decoy_s = np.array(true_s), np.array(decoy_s)
    print("candidates scored: {:,} | hit the max_bonds=34 bail-out: {:,} ({:.1f}%)"
          .format(ntot, bailed, bailed / max(ntot, 1) * 100))
    print("\n{:18} {:>6} {:>9} {:>9} {:>9}".format("", "n", "mean", "median", "p90"))
    for lab, a in [("TRUE candidates", true_s), ("decoys", decoy_s)]:
        if len(a):
            print("{:18} {:>6} {:9.3f} {:9.3f} {:9.3f}"
                  .format(lab, len(a), a.mean(), np.median(a), np.percentile(a, 90)))
    if len(true_s) and len(decoy_s):
        beat = float(np.mean([(decoy_s > t).mean() for t in true_s]))
        print("\nmean fraction of decoys scoring ABOVE the true candidate: {:.1f}%".format(beat * 100))
        print("  (50% = the feature is pure noise; 0% = perfect separation)")
        print("\nVERDICT: Channel 3 {}".format(
            "is NOT discriminating - treat its 4 features as noise, or rebuild per Idea 2.2."
            if beat > 0.35 else "carries real signal - keep it."))
    print("         {:.1f}% of candidates got only the intact parent mass (max_bonds=34)."
          .format(bailed / max(ntot, 1) * 100))

candidates scored: 1,600 | hit the max_bonds=34 bail-out: 244 (15.2%)

                        n      mean    median       p90
TRUE candidates        40     0.472     0.475     0.769
decoys               1560     0.335     0.321     0.571

mean fraction of decoys scoring ABOVE the true candidate: 32.2%
  (50% = the feature is pure noise; 0% = perfect separation)

VERDICT: Channel 3 carries real signal - keep it.
         15.2% of candidates got only the intact parent mass (max_bonds=34).


---
## D5 - Slot waste from tautomer-canonical duplicates

The metric passes both prediction and answer through RDKit tautomer canonicalisation, then
compares the first InChIKey block. Two candidate SMILES that are distinct rows in the pool can
collapse to the **same** key - and each collapse is a submitted slot that can never score.

This measures the collapse rate on real 25-candidate windows.


In [9]:
if not HAVE_RDKIT:
    print("RDKit unavailable - skipping D5.")
else:
    enum = rdMolStandardize.TautomerEnumerator()
    def metric_key(smi):
        m = Chem.MolFromSmiles(smi)
        if m is None: return None
        try:
            return Chem.MolToInchiKey(enum.Canonicalize(m))[:14]
        except Exception:
            return None

    tot_slots, tot_uniq, affected, nq = 0, 0, 0, 0
    for _, r in qg.sample(min(60, len(qg)), random_state=1).iterrows():
        kw, _ = window_keys(r.target, 8.5)
        smis = [co_smi.get(k) or smi_of.get(k) for k in list(kw)[:25]]
        smis = [s for s in smis if s]
        if len(smis) < 5: continue
        keys = [k for k in (metric_key(s) for s in smis) if k]
        if not keys: continue
        nq += 1; tot_slots += len(keys)
        uq = len(set(keys)); tot_uniq += uq; affected += (uq < len(keys))

    if nq:
        waste = (tot_slots - tot_uniq) / tot_slots * 100
        print("windows examined          : {}".format(nq))
        print("slots filled              : {:,}".format(tot_slots))
        print("distinct canonical keys   : {:,}".format(tot_uniq))
        print("WASTED slots              : {:,}  ({:.2f}%)".format(tot_slots - tot_uniq, waste))
        print("windows with >=1 collapse : {} ({:.0f}%)".format(affected, affected / nq * 100))
        print("\nVERDICT: {}".format(
            "Idea 1.4 is worth doing - dedup on the metric key and backfill from rank 26+."
            if waste > 0.5 else "negligible. Skip Idea 1.4."))
    else:
        print("no usable windows sampled.")

windows examined          : 58
slots filled              : 1,289
distinct canonical keys   : 1,282
WASTED slots              : 7  (0.54%)
windows with >=1 collapse : 7 (12%)

VERDICT: Idea 1.4 is worth doing - dedup on the metric key and backfill from rank 26+.


---
## Summary

Copy the `VERDICT:` lines out and use them to order the work. The two that most often change the
plan:

- **D1** - if pool recall at 8.5 ppm is well under 90%, database expansion (Idea 2.1) dominates
  everything else, because no ranker can retrieve what is not in the window.
- **D4** - if a large natural-product library is invisible at 8.5 ppm, Idea 1.1 is about an hour
  of work for a channel-wide recovery.

**Deliberately not covered here:** D2 (rank of the truth under each of the four channels
separately) needs the full pipeline including FPNet, so it belongs in a fork of the baseline
notebook rather than in this standalone diagnostic.

**One caveat on interpretation:** every number here is measured on `enveda-np-examples`, which is
a *Class-1* proxy - the molecule has reference spectra somewhere in the library. It tells you
nothing directly about Classes 2 and 3, where the competition is actually decided. Use it to
catch mechanical defects, not to predict your private score.
